In [3]:
import sys
sys.path.append('..')
import os
import cv2
import rasterio
import numpy as np
from src.utils import tile_image, load_sentinel_rgb_from_files

In [4]:
# Define Paths for Raw Sentinel-2 Bands (.jp2)
season_a_b04 = "../data/raw/season_a/T36UYA_20190904T083601_B04.jp2"
season_a_b03 = "../data/raw/season_a/T36UYA_20190904T083601_B03.jp2"
season_a_b02 = "../data/raw/season_a/T36UYA_20190904T083601_B02.jp2"

season_b_b04 = "../data/raw/season_b/T36UYA_20190318T083701_B04.jp2"
season_b_b03 = "../data/raw/season_b/T36UYA_20190318T083701_B03.jp2"
season_b_b02 = "../data/raw/season_b/T36UYA_20190318T083701_B02.jp2"

In [5]:
# Load, Stack and Normalize RGB Composites
print("Reading and assembling RGB images...")
img_a = load_sentinel_rgb_from_files(season_a_b04, season_a_b03, season_a_b02)
img_b = load_sentinel_rgb_from_files(season_b_b04, season_b_b03, season_b_b02)

print(f"Initial size of Season A: {img_a.shape}")
print(f"Initial size of Season B: {img_b.shape}")

# Spatial Alignment & Resizing
min_h = min(img_a.shape[0], img_b.shape[0])
min_w = min(img_a.shape[1], img_b.shape[1])

img_a = cv2.resize(img_a, (min_w, min_h))
img_b = cv2.resize(img_b, (min_w, min_h))

Reading and assembling RGB images...
Initial size of Season A: (10980, 10980, 3)
Initial size of Season B: (10980, 10980, 3)


In [6]:
# Tiling Large Images into Patches
out_dir_a = "../data/processed/season_a"
out_dir_b = "../data/processed/season_b"
os.makedirs(out_dir_a, exist_ok=True)
os.makedirs(out_dir_b, exist_ok=True)

tiles_a = tile_image(img_a, tile_size=512, overlap=64)
tiles_b = tile_image(img_b, tile_size=512, overlap=64)

# Save Processed Image Patches to Disk
max_patches = min(len(tiles_a), 200)

for idx in range(max_patches):
    tile_a, _ = tiles_a[idx]
    tile_b, _ = tiles_b[idx]

    fname = f"patch_{idx:04d}.png"
    # Convert RGB to BGR for proper OpenCV saving format
    cv2.imwrite(os.path.join(out_dir_a, fname), cv2.cvtColor(tile_a, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(out_dir_b, fname), cv2.cvtColor(tile_b, cv2.COLOR_RGB2BGR))

print(f"generated and saved {max_patches} set of patches in the data/processed folder")

generated and saved 200 set of patches in the data/processed folder
